# Model Monitoring Dashboard

This notebook monitors the end-to-end Craigslist pricing pipeline using the canonical dataset at `structured-v2/datasets/listings_master.csv` and the model artifacts written under `structured-v2/modeling/`.

It is designed to answer three questions as the dataset grows:
1. How accurate is the model over time?
2. Which features matter most right now?
3. Are the model's learned patterns changing as new listings arrive?

In [ ]:
import io
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import storage

PROJECT_ID = os.getenv('PROJECT_ID', '')
BUCKET_NAME = os.getenv('GCS_BUCKET', 'myscrapers_nia18004_v2')
DATA_KEY = os.getenv('DATA_KEY', 'structured-v2/datasets/listings_master.csv')
OUTPUT_PREFIX = os.getenv('OUTPUT_PREFIX', 'structured-v2/modeling')
TIMEZONE = os.getenv('TIMEZONE', 'America/New_York')

storage_client = storage.Client(project=PROJECT_ID or None)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def read_text(bucket_name: str, key: str) -> str:
    return storage_client.bucket(bucket_name).blob(key).download_as_text()

def read_bytes(bucket_name: str, key: str) -> bytes:
    return storage_client.bucket(bucket_name).blob(key).download_as_bytes()

def read_csv(bucket_name: str, key: str) -> pd.DataFrame:
    return pd.read_csv(io.BytesIO(read_bytes(bucket_name, key)))

manifest = json.loads(read_text(BUCKET_NAME, f'{OUTPUT_PREFIX}/latest_manifest.json'))
metrics_history = read_csv(BUCKET_NAME, f'{OUTPUT_PREFIX}/metrics_history.csv')
dataset = read_csv(BUCKET_NAME, DATA_KEY)
latest_predictions = read_csv(BUCKET_NAME, manifest['gcs_paths']['predictions'].replace(f'gs://{BUCKET_NAME}/', ''))
latest_importance = read_csv(BUCKET_NAME, manifest['gcs_paths']['permutation_importance'].replace(f'gs://{BUCKET_NAME}/', ''))

metrics_history['run_ts'] = pd.to_datetime(metrics_history['run_ts'], utc=True, errors='coerce')
dataset['scraped_at'] = pd.to_datetime(dataset['scraped_at'], utc=True, errors='coerce')
dataset['date_local'] = dataset['scraped_at'].dt.tz_convert(TIMEZONE).dt.date
latest_predictions.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

metric_cols = ['mae', 'mape', 'rmse', 'bias']
for metric in metric_cols:
    axes[0, 0].plot(metrics_history['run_ts'], metrics_history[metric], marker='o', label=metric.upper())
axes[0, 0].set_title('Model Accuracy Over Time')
axes[0, 0].set_xlabel('Run Timestamp')
axes[0, 0].legend()

top_importance = latest_importance.head(10).sort_values('importance_mean')
axes[0, 1].barh(top_importance['feature'], top_importance['importance_mean'])
axes[0, 1].set_title('Latest Permutation Importance')
axes[0, 1].set_xlabel('Importance Mean')

daily_prices = dataset.assign(price_num=pd.to_numeric(dataset['price'], errors='coerce')).dropna(subset=['price_num'])
daily_summary = daily_prices.groupby('date_local').agg(median_price=('price_num', 'median'), avg_mileage=('mileage', 'mean'), listings=('post_id', 'count')).reset_index()
axes[1, 0].plot(daily_summary['date_local'], daily_summary['median_price'], marker='o', label='Median price')
axes[1, 0].set_title('Price Trend as Dataset Grows')
axes[1, 0].set_xlabel('Date')
axes[1, 0].legend()

axes[1, 1].plot(metrics_history['run_ts'], metrics_history['train_rows'], marker='o', label='Train rows')
axes[1, 1].plot(metrics_history['run_ts'], metrics_history['holdout_rows'], marker='o', label='Holdout rows')
axes[1, 1].set_title('Dataset Growth by Training Run')
axes[1, 1].set_xlabel('Run Timestamp')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
pdp_uris = manifest['gcs_paths'].get('pdp', [])
if pdp_uris:
    fig, axes = plt.subplots(1, len(pdp_uris), figsize=(6 * len(pdp_uris), 4))
    if len(pdp_uris) == 1:
        axes = [axes]
    for ax, uri in zip(axes, pdp_uris):
        key = uri.replace(f'gs://{BUCKET_NAME}/', '')
        image = plt.imread(io.BytesIO(read_bytes(BUCKET_NAME, key)), format='png')
        ax.imshow(image)
        ax.axis('off')
        ax.set_title(Path(key).stem.replace('pdp_', 'PDP: '))
    plt.tight_layout()
    plt.show()
else:
    print('No PDP artifacts found in the latest manifest.')

In [ ]:
model_view = dataset.copy()
model_view['price_num'] = pd.to_numeric(model_view['price'], errors='coerce')
model_view['mileage_num'] = pd.to_numeric(model_view['mileage'], errors='coerce')
model_view['year_num'] = pd.to_numeric(model_view['year'], errors='coerce')

state_trend = (
    model_view.dropna(subset=['price_num', 'state'])
    .groupby(['date_local', 'state'])
    .agg(median_price=('price_num', 'median'), listings=('post_id', 'count'))
    .reset_index()
)
top_states = state_trend.groupby('state')['listings'].sum().nlargest(4).index

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for state in top_states:
    subset = state_trend[state_trend['state'] == state]
    axes[0].plot(subset['date_local'], subset['median_price'], marker='o', label=state)
axes[0].set_title('Median Price by State Over Time')
axes[0].legend()

recent = latest_predictions.dropna(subset=['actual_price', 'pred_price']).copy()
recent['abs_error'] = (recent['pred_price'] - recent['actual_price']).abs()
recent.sort_values('abs_error', ascending=False).head(10)[['post_id', 'make', 'model', 'actual_price', 'pred_price', 'abs_error']]

axes[1].scatter(model_view['mileage_num'], model_view['price_num'], alpha=0.2)
axes[1].set_title('Observed Mileage vs Price')
axes[1].set_xlabel('Mileage')
axes[1].set_ylabel('Price')
plt.tight_layout()
plt.show()

## How to Use This Notebook

- Re-run the full notebook after a new training artifact set lands in `structured-v2/modeling/`.
- Use the accuracy panel to watch whether performance improves as training rows increase.
- Use permutation importance and PDPs together: importance shows which features matter, PDPs show how the model responds to them.
- Use the drift charts to see whether price patterns are changing by geography, mileage, or time.